# Label Filtering

This notebook defines and applies the label filtering strategy used by all subsequent model notebooks.
The output is `src/valid_labels.json` — a list of label names that all notebooks load instead of
recomputing the filtering each time.

## Filtering strategy (two steps)

**Step 1 — Remove `nickname` attributes**  
The Fashionpedia taxonomy has 11 supercategories. The `nickname` group (153 attributes) contains
highly specific garment sub-types (e.g. "raglan t-shirt", "sailor shirt") that are extremely rare
in the dataset and not reliably distinguishable from images alone. The remaining 10 supercategories
cover visually grounded properties: pattern, length, fit, silhouette, opening type, etc.

**Step 2 — Remove labels below a minimum frequency threshold (`MIN_FREQ`)**  
Even after removing nicknames, some supercategories (e.g. `neckline type`: 31 occurrences total,
`leather`: 199) have too few examples to train or evaluate on meaningfully. We drop any label
appearing fewer than `MIN_FREQ` times in the training set.

This is **label space design**, not feature engineering — the image input is passed raw to the model.
We are choosing *what to predict*, not how to represent the input.

In [ ]:
import sys
import json
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

sys.path.append('/media/data/mmondo/mmondo/DL_Project/data/sketchy/src')
from sketchy.sketchy_dataset import SketchyDataset

DATASET_ROOT = '/media/data/mmondo/mmondo/DL_Project/data/sketchy'
RESULTS_DIR  = '/media/data/mmondo/mmondo/DL_Project/results'
OUTPUT_PATH  = '/media/data/mmondo/mmondo/DL_Project/src/valid_labels.json'

In [ ]:
train_dataset = SketchyDataset(dataset_root=DATASET_ROOT, split='train', compose_global_sketch=False)
print(f"Train samples: {len(train_dataset)}")

## Step 1 — Remove `nickname` attributes

In [ ]:
all_attrs = train_dataset.fp.dataset['attributes']
attr_supercat = {a['name']: a['supercategory'] for a in all_attrs}

non_nickname_names = {a['name'] for a in all_attrs if a['supercategory'] != 'nickname'}
print(f"Attributes in vocabulary:      {len(all_attrs)}")
print(f"After removing nickname:       {len(non_nickname_names)}")

## Step 2 — Remove labels below minimum frequency

In [ ]:
MIN_FREQ = 50  # labels with fewer occurrences in train are dropped

label_counter = Counter()
for idx in range(len(train_dataset)):
    sample = train_dataset[idx]
    for ann in sample['annotations']:
        if ann['top_level']:
            for label in ann['top_level']:
                label_counter[label] += 1

valid_labels = sorted(
    l for l, cnt in label_counter.items()
    if l in non_nickname_names and cnt >= MIN_FREQ
)

print(f"After removing nickname:       {len(non_nickname_names)} (vocabulary)")
print(f"After MIN_FREQ={MIN_FREQ} filter:      {len(valid_labels)} (present in train)")
print(f"Occurrences retained:          "
      f"{sum(label_counter[l] for l in valid_labels) / sum(label_counter.values()) * 100:.1f}% of total")

## Final label set — breakdown by supercategory

In [ ]:
sc_stats = {}
for label in valid_labels:
    sc = attr_supercat.get(label, 'unknown')
    sc_stats.setdefault(sc, {'labels': [], 'total_occ': 0})
    sc_stats[sc]['labels'].append(label)
    sc_stats[sc]['total_occ'] += label_counter[label]

print(f"{'Supercategory':<45} {'labels':>7}  {'occurrences':>12}")
print("-" * 68)
for sc, stats in sorted(sc_stats.items(), key=lambda x: x[1]['total_occ'], reverse=True):
    print(f"  {sc:<43} {len(stats['labels']):>7}  {stats['total_occ']:>12,}")
print()
print(f"Total: {len(valid_labels)} labels")

# Bar chart coloured by supercategory
unique_scs = sorted(sc_stats, key=lambda s: sc_stats[s]['total_occ'], reverse=True)
sc_color = {sc: plt.cm.tab10(i % 10) for i, sc in enumerate(unique_scs)}

sorted_valid = sorted(valid_labels, key=lambda l: label_counter[l], reverse=True)
colors = [sc_color[attr_supercat[l]] for l in sorted_valid]
counts = [label_counter[l] for l in sorted_valid]

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(sorted_valid, counts, color=colors)
ax.set_xticks(range(len(sorted_valid)))
ax.set_xticklabels(sorted_valid, rotation=90, fontsize=7)
ax.set_ylabel('Frequency (train)')
ax.set_title(f'Final label set: {len(valid_labels)} labels (no nickname, MIN_FREQ={MIN_FREQ})')
legend_elements = [Patch(facecolor=sc_color[sc], label=sc) for sc in unique_scs]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/valid_labels_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Save valid labels to disk

All model notebooks load `valid_labels.json` at the start. Do not hardcode label lists elsewhere.

In [ ]:
with open(OUTPUT_PATH, 'w') as f:
    json.dump(valid_labels, f, indent=2)

print(f"Saved {len(valid_labels)} labels to {OUTPUT_PATH}")
print()
print("Load in other notebooks with:")
print("  import json")
print("  with open('/media/data/mmondo/mmondo/DL_Project/src/valid_labels.json') as f:")
print("      valid_labels = json.load(f)")